# RQ2 — Sample-based reproduction of the evidence tracing

Compares a new tracing of the evidence of a sample of repositories, made with the same prompt used in
the study (`data/rq02/evidence/PROMPT_RQ02_EVIDENCE_TRACING_pt.md`), with the items used in the paper.
The new tracing is generated by `scripts/reproduce_evidence_sample.py` and is stored in
`results/rq02/sample_reproduction/`.

The output of an LLM is not deterministic, and splitting evidence into items involves judgment.
The comparison is therefore made at two levels: the number of valid items per repository and
relationship, and the RQ1 categories to which the items were mapped.

In [1]:
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "collection").exists():
    ROOT = ROOT.parent

LABELS = ROOT / "data/rq02/manual_annotation/rq02_labels_annotator_A.csv"
STUDY = ROOT / "data/rq02/evidence/rq02_evidence_items.csv"
NEW = ROOT / "results/rq02/sample_reproduction/rq02_evidence_items.csv"
INPUT = ROOT / "results/rq02/sample_reproduction/input_sample.csv"
OUT = ROOT / "results/rq02/sample_reproduction"
RELATIONSHIPS = ["conflitantes", "redundantes", "complementares"]
NAME_EN = {"conflitantes": "conflicting", "redundantes": "redundant", "complementares": "complementary"}


def repo_key(value):
    m = re.search(r"github\.com/([^/\s]+)/([^/\s]+)", str(value))
    return (f"{m.group(1)}/{m.group(2)}" if m else str(value).strip()).lower()


labels = pd.read_csv(LABELS)
labels["key"] = labels["repo"].map(repo_key)
marked = labels.set_index("key")[RELATIONSHIPS]
sample = set(pd.read_csv(INPUT)["repo"].map(repo_key))


def valid_items(path):
    it = pd.read_csv(path)
    it["key"] = it["repo"].map(repo_key)
    it = it[it["key"].isin(sample) & it["status"].isin(["OK", "PARCIAL"])]
    it = it[[marked.loc[k, r] == 1 for k, r in zip(it["key"], it["relacao"])]].copy()
    it["relationship"] = it["relacao"].map(NAME_EN)
    return it


study, new = valid_items(STUDY), valid_items(NEW)
print(f"Repositories in the sample: {len(sample)}")
print(f"Valid items — study: {len(study)}, new tracing: {len(new)}")

Repositories in the sample: 8
Valid items — study: 51, new tracing: 49


## Items per repository and relationship

In [2]:
counts = pd.DataFrame({
    "study": study.groupby(["key", "relationship"]).size(),
    "new": new.groupby(["key", "relationship"]).size(),
}).fillna(0).astype(int)
counts.to_csv(OUT / "items_per_repository_relationship.csv")
print(f"Repository × relationship pairs with the same number of items: "
      f"{(counts['study'] == counts['new']).sum()} of {len(counts)}")
counts

Repository × relationship pairs with the same number of items: 13 of 17


study  new
key                                    relationship             
directus/directus                      complementary      5    5
                                       redundant          4    4
heroui-inc/heroui                      complementary      1    2
                                       conflicting        1    1
                                       redundant          3    3
kornia/kornia                          complementary      4    4
                                       redundant          2    2
mampfes/hacs_waste_collection_schedule complementary      3    3
                                       redundant          2    1
meshtastic/firmware                    complementary      7    7
                                       conflicting        2    2
                                       redundant          8    7
remix-run/react-router                 complementary      2    2
rsyslog/rsyslog                        complementary      1    1
slick/slick                            complementary      2    2
                                       conflicting        3    2
                                       redundant          1    1

## RQ1 categories per relationship

For each repository and relationship, compares the set of categories to which the items were
mapped (Jaccard index between the sets of the study and of the new tracing).

In [3]:
def category_sets(items):
    return items.groupby(["key", "relationship"])["categoria_rq01"].apply(set)


cs, cn = category_sets(study), category_sets(new)
keys = sorted(set(cs.index) | set(cn.index))
jac = pd.Series({k: len(cs.get(k, set()) & cn.get(k, set())) / len(cs.get(k, set()) | cn.get(k, set()))
                 for k in keys})
print(f"Mean Jaccard of the categories (repository × relationship): {jac.mean():.2f}")

per_category = pd.DataFrame({
    "study": study.groupby(["relationship", "categoria_rq01"]).size(),
    "new": new.groupby(["relationship", "categoria_rq01"]).size(),
}).fillna(0).astype(int)
per_category.to_csv(OUT / "items_per_relationship_category.csv")
per_category

Mean Jaccard of the categories (repository × relationship): 0.88


study  new
relationship  categoria_rq01                               
complementary ai_rules                               5    6
              code_conventions                       1    2
              doc_metadata                           3    2
              performance_&_resource_management      2    2
              project_overview                       1    1
              project_structure                      0    2
              run_and_build                          2    1
              security                               3    3
              system_design                          6    5
              test_instructions                      1    1
              troubleshooting                        1    1
conflicting   ai_rules                               1    1
              doc_metadata                           1    1
              run_and_build                          4    3
redundant     ai_rules                               0    1
              code_conventions                       3    3
              contribution_guidelines                1    0
              doc_metadata                           1    1
              project_overview                       4    4
              project_structure                      1    1
              run_and_build                          4    4
              security                               2    1
              test_instructions                      3    2
              troubleshooting                        1    1